# Macrocosm photo-z CNN
Thin runner: this notebook **only downloads data and calls the entry point** in `photoz_cnn.py`.
All model / training logic lives in `photoz_cnn.py` (edit it to change the model). 3-fold OOF
outlier finding lives in `cv_outliers.py`.

## 1. Setup + data
Clone the repo (brings `photoz_cnn.py`, `eval.py`, `cv_outliers.py`, `splits/`),
install deps, log into Google, download the catalog.

In [ ]:
import os
os.environ['CUTOUT_SIZE'] = '24'   # sample_v3 is 24x24 (registered+cropped); read at import by eval/photoz_cnn
![ -d /content/CNN-Model ] || git clone -b task-v3 https://github.com/Le-Wagon-Macrocosm/CNN-Model.git /content/CNN-Model
%cd /content/CNN-Model
!pip -q install mlflow
from google.colab import auth; auth.authenticate_user()
!gcloud config set project macrocosm-lewagon -q
!mkdir -p /content/data
!gcloud storage cp gs://macrocosm-lewagon/data/sample_v1/catalog_v1.parquet /content/data/
DATA_DIR = '/content/data'


Download the image shards (sample_v3 = 24px, registered+cropped; ~3.5 GB for all 100, fits a **standard** runtime).


In [ ]:
!gcloud storage cp gs://macrocosm-lewagon/data/sample_v3/images_*.npy /content/data/


## 2. Train
Edit `photoz_cnn.py` to change the model. Paste your MLflow token (ask the team) to log the
run + the **50k-val outlier objids**; without a token it just trains. `train()` loads data into
RAM, trains, scores the fixed 50k val, and (with a token) logs params/metrics/outliers to MLflow.

In [ ]:
from photoz_cnn import train

metrics, model = train(
    data_dir='/content/data',
    crop=24,            # sample_v3 is 24x24 (already registered+cropped); fits standard RAM
    N=None,             # cap #train images (e.g. 150000) if RAM-limited; None = all
    batch=256, lr=3e-4, epochs=50,
    run_name='my-run',
    experiment='oa',    # MLflow experiment name (default 'oa')
    mlflow_token='PASTE_MLFLOW_API_TOKEN_HERE',
)
print(metrics)


## 3. (optional) 3-fold OOF outliers → GCS
Trains 3 models; on sample_v3 (24px) all 550k ~3.2 GB fits standard RAM. `seed` decides the fold split.
Results go to `gs://macrocosm-lewagon/results/cv_outliers/`.


In [ ]:
# from cv_outliers import run
# df = run(seed=0, data_dir='/content/data', crop=24)
# -- or as a script --
# !python cv_outliers.py --seed 0 --data-dir /content/data --crop 24 --out gs://macrocosm-lewagon/results/cv_outliers
